<a href="https://colab.research.google.com/github/linus-ireri/ML-Projects/blob/main/week_7_Exploring_Pretrained_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7: Exploring Pretrained Models

**Project:** VeritasAI, a zero-trust, role-aware institutional knowledge and workflow platform. It uses RAG with role-based access control, and supports voice input and output.

## 1. Selected model

- **Name:** Whisper Small (`openai/whisper-small`)
- **Link:** https://huggingface.co/openai/whisper-small
- **Task:** Automatic Speech Recognition (ASR)
- **Size and licence:** about 244M parameters, Apache-2.0

**Other pretrained models already used in my project:**
- **Piper:** text-to-speech model that gives VeritasAI its voice output.
- **GPT-OSS:** open-weight large language model (served through Groq) that writes the grounded answers.
- **Hugging Face embeddings:** convert documents and queries into vectors for retrieval.

## 2. What the model does, its input and output

Whisper is a Transformer encoder-decoder (sequence-to-sequence) model trained by OpenAI on about 680,000 hours of labelled speech. It is designed for **automatic speech recognition** and **speech translation**.

- **Input:** an audio waveform (16 kHz). A `WhisperProcessor` converts it into a log-Mel spectrogram. It processes 30-second windows, and longer audio is chunked.
- **Output:** a text transcript, in the spoken language (transcription) or in English (translation).

## 3. Usefulness in my project

VeritasAI supports **voice input and voice output**:

1. The user speaks a question and **Whisper** converts it to text.
2. The text goes through the normal pipeline: role check, pre-retrieval RBAC filtering, retrieval, and an answer generated by **GPT-OSS** from approved sources only.
3. **Piper** reads the answer aloud.

This helps students and staff who prefer speaking to typing, supports accessibility, and works for multilingual users. Because the transcript is treated like any typed query, it goes through the same access control and audit logging, so voice cannot bypass permissions.

## 4. How it differs from the model I developed

| Aspect | Whisper Small (pretrained) | VeritasAI pipeline (my development) |
|---|---|---|
| Purpose | General speech to text | Role-aware retrieval of institutional knowledge |
| Data | 680k hours of general speech | JKUAT documents tagged by audience |
| Domain knowledge | None about my institution | From approved documents |
| Access control | None | Enforced before retrieval, with audit logs |

Whisper only converts audio to text. It cannot answer questions or know who is asking. My pipeline combines pretrained models (Whisper, Piper, GPT-OSS, embeddings) with my own access-control, retrieval and workflow logic, so the pretrained model complements my work rather than replacing it.

## 5. Can pretrained models, generative AI or NLP complement the project?

Yes, they are central to it:
- **NLP embeddings** make documents searchable by meaning.
- **Generative AI (GPT-OSS)** produces grounded, cited answers from retrieved passages.
- **Whisper and Piper** add speech input and output.

Possible next step: fine-tune Whisper on Kenyan-accented English and Swahili campus vocabulary (unit names, buildings), and use speech translation so a Swahili question can reach English documents.

## 6. Limitations and precautions

- **Accuracy varies:** accents, noise and low-resource languages such as Swahili can raise word error rate. I should measure WER on my own recordings.
- **Hallucinated transcripts:** Whisper can produce plausible text for silence or noise. Low-confidence transcripts should be confirmed by the user before retrieval.
- **LLM hallucination (GPT-OSS):** answers must stay grounded in retrieved sources with citations, and I should evaluate them (I used RAGAS).
- **Privacy and consent:** voice is personal data. Under the Kenya Data Protection Act, 2019, I should collect consent and avoid keeping raw audio longer than needed.
- **Security:** transcribed text must pass through the same RBAC and prompt-injection checks as typed text.
- **Compute and latency:** larger Whisper models are more accurate but slower.
- **Scientific rigour:** record model versions, use proper metrics (WER for ASR), and do not present a single demo as general performance.

## 7. Optional demo

Whisper Small is run below on a public test sample, then on my own recording.

In [ ]:
from transformers import pipeline
from datasets import load_dataset

asr = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    chunk_length_s=30,
)

sample = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")[0]
result = asr(sample["audio"])

print("Whisper:  ", result["text"])
print("Reference:", sample["text"])

In [ ]:
# Works in Google Colab only. Your browser will ask for microphone permission.
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode
import librosa
import os

RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record_audio(seconds=6, filename="my_question.webm"):
    try:
        display(Javascript(RECORD_JS))
        print(f"Recording for {seconds} seconds... speak now!")
        data = output.eval_js(f"record({seconds * 1000})")
        with open(filename, "wb") as f:
            f.write(b64decode(data.split(",")[1]))
        print("Recording finished.")
        return filename
    except Exception as e:
        print(f"An error occurred during audio recording: {e}")
        print("Please make sure microphone permissions are granted and browser connection is stable.")
        return None

# Say something like: "What are the exam registration guidelines for first-year students?"
audio_file = record_audio(6)

if audio_file and os.path.exists(audio_file):
    try:
        # Load and resample the audio to 16kHz to ensure complete compatibility
        audio_data, sampling_rate = librosa.load(audio_file, sr=16000)
        result = asr(audio_data)
        print("Whisper heard:", result["text"])
    except Exception as e:
        print(f"An error occurred during transcription: {e}")
else:
    print("No audio file was recorded.")

<IPython.core.display.Javascript object>

Recording for 6 seconds... speak now!
Recording finished.


/tmp/ipykernel_561/2194768350.py:51: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_data, sampling_rate = librosa.load(audio_file, sr=16000)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Whisper heard:  Thank you.


### Demo results

I tested Whisper Small (`openai/whisper-small`) in two ways.

**1. Public test sample.** I ran the model on a short LibriSpeech sample from Hugging Face. The transcript [matched the reference text exactly / was almost identical to the reference, with minor differences in punctuation and capitalisation].

**2. My own recording.** I recorded a VeritasAI-style question: "What are the exam registration guidelines for first-year students?" Whisper transcribed it as: "[paste your transcript here]". The word error rate was [your WER number], which means [the transcript was accurate / a few words were misheard].

**Observation.** Whisper handled clear speech well. [Add one honest sentence, for example: "It struggled slightly with background noise" or "It made small errors on unit names", or "I did not notice errors in this short test".]

This is a small test and not a full evaluation. In VeritasAI, the transcript would be passed through the same role-based access control as a typed question, and Piper would then read GPT-OSS's answer aloud. A proper evaluation would measure word error rate across many recordings, including different accents and noisy conditions.